# data incident responder: evaluation run

Everything runs in this notebook: the model is served by vLLM on the GPU, and the warehouse, dbt, the MCP server
and the agents run next to it. Runtime: A100 (80 GB) with high RAM.

Optional: add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` as Colab secrets to trace every run.

Results are written to Drive after each incident. If the runtime drops, run all again and finished incidents are skipped.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/arihaaant/data-incident-responder.git /content/responder
%cd /content/responder
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
/content/responder
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/

In [2]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

tracing on


## serve the model

In [3]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

model up after 8.0 min


## warehouse

In [4]:
!python -m responder.warehouse --path kit/source.duckdb
!python -m pytest -q tests

built kit/source.duckdb
.....FFFFF........FFFFFF....FF..                                         [100%]
=================================== FAILURES ===================================
______________ test_reads_allowed[select * from marts.fct_orders] ______________

q = 'select * from marts.fct_orders'

    @pytest.mark.parametrize("q", [
        "select * from marts.fct_orders",
        "with x as (select 1 as a) select a from x",
        "select order_date, count(*) from staging.stg_orders group by 1 order by 1 desc limit 5",
        "select 1 union all select 2",
        "select table_name from information_schema.tables",
    ])
    def test_reads_allowed(q):
>       assert check_sql(q) is None
               ^^^^^^^^^^^^

tests/test_tools.py:21: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

query = 'select * from marts.fct_orders'

    def check_sql(query):
        """Returns None if the query is a plain read, otherwise the reason it is refused.

## runs: the responder and both baselines on the 48 development incidents

In [6]:
from google.colab import drive

drive.mount("/content/drive")
out = "/content/drive/MyDrive/data-incident-responder/runs"
os.makedirs(out, exist_ok=True)

Mounted at /content/drive


In [7]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.run --system {system} --split dev --workers 6 --out {out}/dev-{system}


=== multi ===
[10/10/26 04:59:46] INFO     Processing request of type            ]8;id=19822;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=818544;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 04:59:46] INFO     Processing request of type            ]8;id=562736;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=223243;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 04:59:46] INFO     Processing request of type            ]8;id=930163;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=314139;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/s

## results

In [8]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.report {out}/dev-{system}


=== multi ===
                   n    root   cause  action  resolved  tokens
overall           47   70.2%   74.5%   70.2%     61.7%   17887
duplicates        11   72.7%   72.7%   81.8%     63.6%   20804
late_data         12   58.3%   58.3%   50.0%     50.0%   20766
null_spike        12   58.3%   66.7%   66.7%     66.7%   21023
schema_drift      12   91.7%  100.0%   83.3%     66.7%    9197
dev               47   70.2%   74.5%   70.2%     61.7%   17887

resolved 29/47 (95% CI 47 to 74%), fixes passing the hidden check 16/33

=== single ===
                   n    root   cause  action  resolved  tokens
overall           40   37.5%   55.0%   80.0%     60.0%   19418
duplicates        11   54.5%   72.7%   72.7%     27.3%   22978
late_data         10   40.0%   50.0%   50.0%     40.0%   22955
null_spike         8   62.5%  100.0%  100.0%    100.0%   23537
schema_drift      11    0.0%    9.1%  100.0%     81.8%    9648
dev               40   37.5%   55.0%   80.0%     60.0%   19418

resolved 24/4